In [1]:
%pip install groq python-dotenv --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 10.4 MB/s eta 0:00:00


In [2]:
# Setup

from dotenv import load_dotenv
load_dotenv()

import os
import getpass
from groq import Groq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "llama-3.1-8b-instant"

Enter your Groq API Key: ··········


In [3]:
# Define Experts

MODEL_CONFIG = {
    "technical": {
        "system_prompt": """You are a Senior Software Engineer and Technical Support Expert.
You are rigorous, precise, and code-focused.
When answering:
- Always identify the root cause first.
- Provide working code snippets where relevant.
- Use technical terminology accurately.
- If you need more info to debug, ask for it specifically.""",
        "temperature": 0.7
    },
    "billing": {
        "system_prompt": """You are an empathetic Billing and Payments Specialist.
You are calm, understanding, and policy-driven.
When answering:
- Always acknowledge the customer's frustration first.
- Explain the refund/billing policy clearly.
- Provide concrete next steps (e.g., 'You will receive a refund in 5-7 business days').
- Never make promises outside company policy.""",
        "temperature": 0.7
    },
    "general": {
        "system_prompt": """You are a friendly and helpful Customer Support Agent.
You handle general inquiries, product questions, and casual conversation.
Keep responses warm, concise, and helpful.""",
        "temperature": 0.7
    },
    # 🚀 BONUS: Tool Use Expert
    "tool": {
        "system_prompt": "You are a data assistant that reports real-time information.",
        "temperature": 0.0
    }
}

In [4]:
# The Router

def route_prompt(user_input: str) -> str:
    """
    Classifies user input into one of: technical, billing, general, tool.
    Uses temperature=0 for consistency (deterministic output).
    """
    routing_prompt = f"""Classify the following user message into EXACTLY ONE of these categories:
- technical   (bug reports, code errors, software issues)
- billing     (charges, refunds, subscriptions, payments)
- tool        (requests for live/real-time data like prices, weather, stocks)
- general     (everything else)

Return ONLY the single category word. No punctuation. No explanation.

User message: "{user_input}"
Category:"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": routing_prompt}],
        temperature=0.0,
        max_tokens=10  # We only need one word
    )

    category = response.choices[0].message.content.strip().lower()

    # Safety fallback — if the model returns something unexpected
    if category not in MODEL_CONFIG:
        category = "general"

    return category

In [5]:
# Mock Tool Function

def mock_fetch_data(user_input: str) -> str:
    """
    Simulates fetching real-time data.
    In a real system, this would call a live API.
    """
    mock_data = {
        "bitcoin":  "Bitcoin (BTC) is currently trading at $67,432 USD.",
        "ethereum": "Ethereum (ETH) is currently trading at $3,512 USD.",
        "weather":  "Current weather in New York: 72°F, partly cloudy.",
    }

    user_lower = user_input.lower()
    for keyword, data in mock_data.items():
        if keyword in user_lower:
            return f"[MOCK DATA] {data}"

    return "[MOCK DATA] Live data fetched: $42,000 USD (default mock response)."

In [6]:
# The Orchestrator

def process_request(user_input: str) -> str:
    """
    Main orchestrator:
    1. Routes the query to the right expert category.
    2. If 'tool' category, calls mock data function instead of LLM.
    3. Otherwise, calls LLM with the expert's system prompt.
    """
    # Step 1: Route
    category = route_prompt(user_input)
    print(f"  → Routed to: [{category.upper()} EXPERT]")

    # Step 2: Bonus tool path
    if category == "tool":
        return mock_fetch_data(user_input)

    # Step 3: Load the right expert config
    config = MODEL_CONFIG[category]

    # Step 4: Call the LLM with expert system prompt
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": config["system_prompt"]},
            {"role": "user",   "content": user_input}
        ],
        temperature=config["temperature"]
    )

    return response.choices[0].message.content.strip()

In [8]:
# Test

test_queries = [
    "My Python script is throwing an IndexError on line 5.",
    "I was charged twice for my subscription this month.",
    "What are your business hours?",
    "What is the current price of Bitcoin?",  # Bonus tool use
]

print("=" * 55)
print("       SMART CUSTOMER SUPPORT ROUTER (MoE)")
print("=" * 55)

for query in test_queries:
    print(f"\n USER: {query}")
    answer = process_request(query)
    print(f" EXPERT: {answer}")
    print("-" * 55)


       SMART CUSTOMER SUPPORT ROUTER (MoE)

 USER: My Python script is throwing an IndexError on line 5.
  → Routed to: [TECHNICAL EXPERT]
 EXPERT: An `IndexError` typically occurs when you're trying to access an element in a list or string at an index that doesn't exist.

To help you further, could you please provide the following:

1. The code snippet where the error is occurring (line 5).
2. The entire code snippet (if it's not too long).
3. The full error message, including the line number and any other relevant information.

Additionally, please provide some context about how your script is working. What's the expected input and output? Are you iterating over a list or string?

With this information, I'll be able to help you identify the root cause of the `IndexError` and provide a solution.

Also, please check the following:

- Is the list or string you're trying to access empty?
- Are you trying to access an index that's out of range?

Here's a simple example of how you might ge